In [1]:
import torch
import torch.backends.cudnn as cudnn

import numpy as np

from utils.augmentations import letterbox

from models.common import DetectMultiBackend
from utils.general import (check_img_size, cv2, non_max_suppression, scale_coords, xyxy2xywh)
from utils.plots import Annotator, colors
from utils.segment.general import process_mask, scale_masks
from utils.segment.plots import plot_masks

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
half = device.type != 'cpu'

C:\Users\Owner\anaconda3\envs\yolo_seg\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loads the models with correct parameters which can be resued
def loadModel(weights, imgsz=640):
    model = DetectMultiBackend(weights, device=device, dnn=False, data='data/coco128.yaml', fp16=half)
    imgsz = check_img_size(imgsz, s=model.stride)  # check image size
    
    return model


### Not sure how this section of code works or what it does exactly
# Prepares image for introduction to model for return of predictions
def img_prep(img, imgsz, stride):
    img = letterbox(img, imgsz, stride=stride)[0]
    
    img = img[:,:,::-1].transpose(2, 0, 1)
    img = np.ascontiguousarray(img)
    
    img = torch.from_numpy(img).to(device)
    img = img.half() if half else img.float()  # uint8 to fp16/32
    img /= 255.0  # 0 - 255 to 0.0 - 1.0
    if img.ndimension() == 3:
        img = img.unsqueeze(0)
        
    return img


# Actually runs the detections and returns both mask and detection locations 
def detectObjects(
        im0,
        model,
        imgsz=(640, 640),  # inference size (height, width)
        conf_thres=0.25,  # confidence threshold
        iou_thres=0.45,  # NMS IOU threshold
        view_img=True,  # show results
        classes=None,  # filter by class: --class 0, or --class 0 2 3
        nameAdd="",
        maskModel=True
):  
    height_img,width_img,ch_img = im0.shape
    
    # Prepare introduced image
    im = img_prep(im0, imgsz, model.stride)
    
    # Feed image to model for returned results
    with torch.no_grad():
        if maskModel:
            pred, out = model(im)
            proto = out[1]
        else:
            pred = model(im)
    
    # Parse through detections to finalize results
    det = non_max_suppression(pred, conf_thres, iou_thres, classes, nm=32)[0]

    gn = torch.tensor(im0.shape)[[1, 0, 1, 0]]  # normalization gain whwh

    annotator = Annotator(np.ascontiguousarray(im0), line_width=3, example=str(model.names))

    if len(det):
        if maskModel:
            # Process mask from split in detection
            masks = process_mask(proto[0], det[:, 6:], det[:, :4], im.shape[2:], upsample=True)  # HWC
        
            mcolors = [colors(int(6), True) for cls in det[:, 5]]
            im_masks = plot_masks(im[0], masks, mcolors)  # image with masks shape(imh,imw,3)
            annotator.im = scale_masks(im.shape[2:], im_masks, im0.shape)  # scale to original h, w

        # Scale cordinates for viewing image if decided
        det[:, :4] = scale_coords(im.shape[2:], det[:, :4], im0.shape).round()
        
        # Sorts and organizes detections by class
        det = det[:, :6]
        det = torch.stack(sorted(det, key=lambda det : det[-1]))
        
        output = []
        for *xyxy, conf, cls in det:
            
            xywh = (xyxy2xywh(torch.tensor(xyxy).view(1, 4)) / gn).view(-1).tolist()
            
            # Append normalized data to output
            output.append([cls.item(), *xywh]) # , xywh[2]*xywh[3]]) 
            
            # If viewing image is selected than annotate image
            if view_img:  # Add bbox to image
                c = int(cls)  # integer class
                label = None if False else (model.names[c] if False else f'{model.names[c]} {conf:.2f}')
                annotator.box_label(xyxy, label, color=colors(c, True))
    
    else:
        
        if view_img:
            cv2.imshow(f"Image{nameAdd}", cv2.resize(im0, (512,512)))
            cv2.waitKey(1)  # 1 millisecond
        
        # If nothing is detected return None for both mask and output
        return None, None
            
    # Stream results
    im0 = annotator.result()
    
    if view_img:
        cv2.imshow(f"Image{nameAdd}", cv2.resize(im0, (512,512)))
        cv2.waitKey(1)  # 1 millisecond
    
    if maskModel:
        return masks, output
    else:
        return output

In [3]:
# # SAMPLE CODE FOR TESTING

# img = cv2.imread("stereo2.jpg")

# weight = "yolov7-tiny.pt"
# weight = "yolov5s-seg.pt"
# model = loadModel(weight)

# # img = cv2.resize(img, (640, 640))

# masks, det = detectObjects(img, model, classes=[0, 58] ,conf_thres=0.6, view_img=True, maskModel=True)
# print(det)


# cv2.imshow("Ime", img )
# cv2.waitKey(1)  

Fusing layers... 
YOLOv5s-seg summary: 224 layers, 7611485 parameters, 0 gradients, 26.4 GFLOPs


[[0.0, 0.488696813583374, 0.5729166865348816, 0.07845744490623474, 0.375, 0.029421541839838028], [58.0, 0.19680851697921753, 0.5020833611488342, 0.11702127754688263, 0.2666666805744171, 0.03120567564000476]]


-1